In [1]:
import os
import re
import time
import sqlite3
import numpy as np
import pandas as pd
import requests

from tqdm import tqdm
from dotenv import load_dotenv

In [6]:
# 데이터 로드
RAW_PATH = "../data/naviya_raw_20260126_130108.csv"
ST_PATH  = "../data/서울교통공사_역주소 및 전화번호_20250318.csv"
CODE_PATH = "../data/sigungu_code_info.csv"

df_raw = pd.read_csv(RAW_PATH)
df = df_raw.copy()

df["name"] = df["name_raw"]
df["category"] = df["category_raw"]

# 서교공 파일은 EUC-KR 인코딩
st_raw = pd.read_csv(ST_PATH, encoding="euc-kr")

print(df_raw.shape, df_raw.columns.tolist())
print(st_raw.shape, st_raw.columns.tolist())

(352, 10) ['source_site', 'crawled_at', 'listing_url', 'detail_url', 'external_id', 'name_raw', 'category_raw', 'address_raw', 'map_url_raw', 'status_raw']
(289, 7) ['연번', '역번호', '호선', '역명', '역전화번호', '도로명주소', '지번주소']


In [7]:
#주소 표준화
def normalize_spaces(x: str) -> str:
    x = str(x).strip()
    x = re.sub(r"[《》]", " ", x)          # 특수 괄호류 제거
    x = re.sub(r"\s+", " ", x).strip()
    return x

def normalize_seoul_prefix(addr: str) -> str:
    """
    '서울', '서울시' -> '서울특별시'로 통일
    """
    if addr is None or (isinstance(addr, float) and np.isnan(addr)):
        return addr
    a = normalize_spaces(addr)
    a = re.sub(r"^서울시\s+", "서울특별시 ", a)
    a = re.sub(r"^서울\s+", "서울특별시 ", a)
    return a

In [8]:
# 역 매핑 테이블 생성
def clean_station_name(name: str) -> str:
    """
    서교공 역명: '종로3가' -> '종로3가역'
    """
    n = normalize_spaces(name)
    n = re.sub(r"\(.*?\)", "", n).strip()  # 혹시 괄호 부가정보가 있으면 제거
    if not n.endswith("역"):
        n += "역"
    return n

st = st_raw.copy()

st["station_std"] = st["역명"].apply(clean_station_name)

# 서교공 도로명주소는 대개 '서울특별시 ...' 형태지만, 안전하게 prefix normalize
st["station_road_addr"] = st["도로명주소"].astype(str).apply(normalize_seoul_prefix)

# 대표 주소 선택(환승역 중복 제거)
st_master = (
    st.groupby("station_std", as_index=False)
      .agg(station_road_addr=("station_road_addr", "first"))
)

station_road_dict = dict(zip(st_master["station_std"], st_master["station_road_addr"]))
station_master_list = st_master["station_std"].tolist()

In [22]:
# ============ 주소 파싱 ===============
df = df_raw.copy()

# name/category는 raw 그대로
df["name"] = df["name_raw"]
df["category"] = df["category_raw"]

def normalize_seoul_prefix(addr: str) -> str:
    if addr.startswith("서울 "):
        return addr.replace("서울 ", "서울특별시 ", 1)
    return addr

def extract_core_address(address_raw: str):
    if pd.isna(address_raw):
        return None
    
    # 1. 슬래시 기준 분리
    parts = [p.strip() for p in address_raw.split("/") if p.strip()]
    
    # 2. 서울 + 구 포함 세그먼트만 필터
    seoul_parts = [
        normalize_seoul_prefix(p)
        for p in parts
        if "서울" in p and "구" in p
    ]
    
    if not seoul_parts:
        return None
    
    # 3. 도로명주소가 괄호 안에 있는 경우 우선 사용
    for p in seoul_parts:
        match = re.search(r"\((.*?)\)", p)
        if match:
            road = match.group(1).strip()
            if "로" in road or "길" in road:
                # 구/동까지는 앞에서 추출
                prefix = re.split(r"\(", p)[0].strip()
                return prefix + " " + road
    
    # 4. 도로명 단독 세그먼트가 있는 경우
    for p in seoul_parts:
        if "로" in p or "길" in p:
            return p
    
    # 5. 없으면 첫 번째 서울 주소 반환
    return seoul_parts[0]

df["address_parsed"] = df["address_raw"].apply(extract_core_address)
df["address_std"] = df["address_parsed"]
df.drop(columns=["address_parsed"], inplace=True)


df[["address_raw", "address_std"]].head(10)

,address_raw,address_std
0,서울특별시 양천구 신정동 1031-9 (중앙로 262) / 신정네거리역 4번출구 도보1분,서울특별시 양천구 신정동 1031-9 중앙로 262
1,서울특별시 관악구 남현동 1061-24/서울특별시 관악구 과천대로 947-2/사당역...,서울특별시 관악구 과천대로 947-2
2,서울특별시 마포구 도화동 1-12 (새창로 40) / 공덕역 10번출구 도보2분,서울특별시 마포구 도화동 1-12 새창로 40
3,서울특별시 성동구 도선동 58-1 (왕십리로 326)/왕십리역 1번출구 도보5분 (...,서울특별시 성동구 도선동 58-1 왕십리로 326
4,서울특별시 강서구 마곡동 797-10 (마곡동로4길 15) / 발산역 9번출구 도보...,서울특별시 강서구 마곡동 797-10 마곡동로4길 15
5,서울특별시 중랑구 상봉동 81-5/서울특별시 중랑구 망우로 346/상봉역 2번출구 ...,서울특별시 중랑구 망우로 346
6,서울특별시 금천구 가산동 41-11(벚꽃로 312) / 가산디지털단지역 2번출구 도보2분,서울특별시 금천구 가산동 41-11 벚꽃로 312
7,서울특별시 동대문구 제기동 / 제기동역 5번출구 도보5분,서울특별시 동대문구 제기동
8,서울특별시 광진구 자양동 52-65/서울특별시 광진구 뚝섬로 476/영동대교 사거리...,서울특별시 광진구 뚝섬로 476
9,서울특별시 구로구 구로동 187-3/서울특별시 구로구 도림천로 448/구로디지털단지...,서울특별시 구로구 구로동 187-3


In [23]:
# ========= is_valid_location 판단 =============
def normalize_spaces(text):
    if pd.isna(text):
        return None
    return re.sub(r"\s+", " ", str(text)).strip()


def looks_like_address(text):
    if text is None:
        return False
    
    # 서울 + 구 포함 여부
    if "서울" in text and "구" in text:
        return True
    
    return False

def is_station_based_raw(raw: str):
    """
    raw 주소가 실제 업소 주소가 아니라
    '신림역 2번출구 도보2분' 같은 역 기반 설명이면 True 반환
    """
    if raw is None or (isinstance(raw, float) and np.isnan(raw)):
        return False

    t = normalize_spaces(raw)

    # 1) 아예 주소 형식이 아닌 경우
    if not looks_like_address(t):
        return True

    # 2) 주소는 있는데 괄호 뒤가 전부 역 설명인 경우
    if re.search(r"(역|출구|도보)", t) and not re.search(r"\d+로|\d+길|\d+-\d+", t):
        # 도로명 숫자 패턴이 없고 역 키워드가 중심이면
        return True

    return False


df["is_station_based_raw"] = df["address_raw"].apply(is_station_based_raw)

print("역 기반 raw 주소 개수:", df["is_station_based_raw"].sum())

역 기반 raw 주소 개수: 136


In [25]:
def extract_gu(addr):
    if pd.isna(addr):
        return None
    match = re.search(r"서울특별시\s+(\S+구)", addr)
    if match:
        return match.group(1)
    return None

df["gu"] = df["address_std"].apply(extract_gu)

In [26]:
# station_detail 파싱
def extract_station_detail(address_raw):
    if pd.isna(address_raw):
        return np.nan
    
    parts = [p.strip() for p in str(address_raw).split("/") if p.strip()]
    
    for p in parts:
        if "역" in p:
            return p
    
    return np.nan

df["station_detail"] = df["address_raw"].apply(extract_station_detail)

In [27]:
# station_exit 및 station 파싱
def find_station_by_master(text: str):
    if text is None or (isinstance(text, float) and np.isnan(text)):
        return None
    t = normalize_spaces(text)

    # 포함되는 역 후보 찾기
    matches = [s for s in station_master_list if s in t]
    if not matches:
        return None

    # 가장 먼저 등장하는 역을 채택
    matches.sort(key=lambda s: t.index(s))
    return matches[0]

def parse_station_fields_with_master(station_detail):
    if station_detail is None or (isinstance(station_detail, float) and np.isnan(station_detail)):
        return (np.nan, np.nan)

    t = normalize_spaces(station_detail)

    # 도보 n분 제거(출구/역명만 남김)
    t2 = re.sub(r"도보\s*\d+\s*분", "", t).strip()

    station = find_station_by_master(t2)
    if not station:
        return (np.nan, np.nan)

    m_exit = re.search(r"(\d+)\s*번\s*출구", t2)
    if m_exit:
        station_exit = f"{station} {m_exit.group(1)}번출구"
    else:
        station_exit = np.nan

    return (station_exit, station)

df[["station_exit", "station"]] = (
    df["station_detail"]
    .apply(lambda x: pd.Series(parse_station_fields_with_master(x)))
)

display(df[["station_detail","station_exit","station"]].sample(20, random_state=2))

,station_detail,station_exit,station
25,성수역 2번출구 도보2분,성수역 2번출구,성수역
225,선릉역 1번출구 도보3분,선릉역 1번출구,선릉역
240,신사역 4번출구 도보2분,신사역 4번출구,신사역
267,문정역 3번출구 도보3분,문정역 3번출구,문정역
243,내방역 7번출구 도보1분,내방역 7번출구,내방역
339,문정역 3번출구 도보3분,문정역 3번출구,문정역
322,이수역 3번출구 도보2분,이수역 3번출구,이수역
84,수유역 5번출구 도보3분,수유역 5번출구,수유역
69,태릉입구역 6번출구 도보5분,태릉입구역 6번출구,태릉입구역
68,마곡역 4번출구 도보2분,마곡역 4번출구,마곡역


In [20]:
print(df.columns.tolist())

['source_site', 'crawled_at', 'listing_url', 'detail_url', 'external_id', 'name_raw', 'category_raw', 'address_raw', 'map_url_raw', 'status_raw', 'name', 'category', 'address_parsed', 'is_station_based_raw', 'gu', 'station_detail', 'station_exit', 'station']


In [28]:
# 자치구 파싱
def parse_gu_from_addr(addr: str):
    if addr is None or (isinstance(addr, float) and np.isnan(addr)):
        return np.nan
    a = normalize_seoul_prefix(addr)
    m = re.search(r"서울특별시\s+(\S+구)\b", a)
    if m:
        return m.group(1)
    # 혹시 '서울 강남구' 형태가 남아있다면
    m2 = re.search(r"서울\s+(\S+구)\b", a)
    if m2:
        return m2.group(1)
    return np.nan

# 1차 gu 채우기(상세주소가 있으면 주소에서)
df["gu"] = df["address_std"].apply(parse_gu_from_addr)

In [29]:
def is_valid_location(addr):
    if pd.isna(addr):
        return False
    return addr.startswith("서울특별시") and "구" in addr

df["is_valid_location"] = df["address_std"].apply(is_valid_location)

In [30]:
load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
print(GOOGLE_API_KEY[:6] + "..." if GOOGLE_API_KEY else "❌ 키 로드 실패")

AIzaSy...


In [31]:
# geocoding fuction
def geocode_google(query: str, api_key: str):
    url = "https://maps.googleapis.com/maps/api/geocode/json"
    params = {
        "address": query,
        "key": api_key,
        "language": "ko",
        "region": "kr",
    }
    r = requests.get(url, params=params, timeout=20)
    data = r.json()

    if data.get("status") != "OK" or not data.get("results"):
        return (None, None, None, None)

    top = data["results"][0]
    loc = top["geometry"]["location"]
    lat, lng = loc["lat"], loc["lng"]

    # gu 추출(가능한 범위 내)
    gu = None
    for comp in top.get("address_components", []):
        if "sublocality_level_1" in comp.get("types", []):
            gu = comp.get("long_name")
            break
        if "administrative_area_level_2" in comp.get("types", []):
            gu = comp.get("long_name")

    return (lat, lng, gu, top.get("formatted_address"))

cache = {}

def geocode_cached(query: str, sleep_sec=0.1):
    if query is None or (isinstance(query, float) and np.isnan(query)):
        return (None, None, None, None)
    q = normalize_spaces(query)
    if not q:
        return (None, None, None, None)
    if q in cache:
        return cache[q]

    out = geocode_google(q, GOOGLE_API_KEY)
    cache[q] = out
    time.sleep(sleep_sec)
    return out

def build_station_exit_query(station_exit: str):
    # 서울로 유도
    return f"서울특별시 {station_exit}" if station_exit else None

def build_station_query(station: str):
    # 1) 서교공 도로명주소가 있으면 그걸 최우선 사용
    if station in station_road_dict:
        return station_road_dict[station]
    # 2) 없으면 '서울특별시 + 역명'으로 보완
    return f"서울특별시 {station}" if station else None

In [32]:
# run geocoding

df["lat"] = np.nan
df["lng"] = np.nan
df["COL_ADM_SE"] = np.nan

# is_valid_location은 지오코딩 성공 여부가 아니라
# "실제 업소 주소인지 여부"로 먼저 결정
df["is_valid_location"] = ~df["is_station_based_raw"]

# 1) 실제 업소 주소인 경우만 address_std로 1차 지오코딩
for i, row in df.iterrows():
    if row["is_valid_location"] and pd.notna(row["address_std"]):
        lat, lng, gu_geo, _ = geocode_cached(row["address_std"])
        if lat is not None:
            df.at[i, "lat"] = lat
            df.at[i, "lng"] = lng

            if pd.isna(df.at[i, "gu"]) and gu_geo:
                df.at[i, "gu"] = gu_geo

# 2) 역 기반인 경우는 항상 is_valid_location=False 유지
#    station_exit → station fallback 수행

mask_station_based = df["is_valid_location"] == False

# station_exit 우선
for i, row in df[mask_station_based].iterrows():
    if pd.notna(row["station_exit"]):
        q = build_station_exit_query(row["station_exit"])
        lat, lng, gu_geo, _ = geocode_cached(q)
        if lat is not None:
            df.at[i, "lat"] = lat
            df.at[i, "lng"] = lng

            if pd.isna(df.at[i, "gu"]) and gu_geo:
                df.at[i, "gu"] = gu_geo

# 여전히 실패 → station (서교공 도로명주소)
mask_need2 = (df["is_valid_location"] == False) & (df["lat"].isna())

for i, row in df[mask_need2].iterrows():
    if pd.notna(row["station"]):
        q = build_station_query(row["station"])
        lat, lng, gu_geo, _ = geocode_cached(q)
        if lat is not None:
            df.at[i, "lat"] = lat
            df.at[i, "lng"] = lng

            if pd.isna(df.at[i, "gu"]):
                df.at[i, "gu"] = parse_gu_from_addr(q)
            if pd.isna(df.at[i, "gu"]) and gu_geo:
                df.at[i, "gu"] = gu_geo

In [33]:
# gu 최종 보완
# gu가 남아있으면 raw에서라도 파싱 시도
mask_gu_missing = df["gu"].isna()
df.loc[mask_gu_missing, "gu"] = df.loc[mask_gu_missing, "address_raw"].apply(parse_gu_from_addr)

print("gu 결측 개수:", df["gu"].isna().sum())
display(df[df["gu"].isna()][["name","address_raw","address_std","station_detail","station_exit","station"]].head(30))

gu 결측 개수: 2


,name,address_raw,address_std,station_detail,station_exit,station
177,구로디지털1인샵[휴] ♡ 손 끝으로 전하는 감동 ■// 건식 //■// 림프순환테라...,서울시 동작구 신대방동 719(신대방1가길 38)/구로디지털단지역 6번출구 도보5분,서울시 동작구 신대방동 719 신대방1가길 38,구로디지털단지역 6번출구 도보5분,구로디지털단지역 6번출구,구로디지털단지역
218,"홈케어[1%명품] ❤️❤️24시❤️❤️ㅣ강남구ㅣ강남,논현,신사,선릉,삼성,역삼","홈케어: 강남구 (강남,논현,신사,선릉,삼성,역삼)",None,"홈케어: 강남구 (강남,논현,신사,선릉,삼성,역삼)",NaN,NaN


In [34]:
# COL_ADM_SE 매핑
code_df = pd.read_csv(CODE_PATH, encoding="utf-8", dtype=str)

# 서울(시도코드 11)만 필터
seoul_gu = code_df[code_df["SIGUNGU_CD"].str.startswith("11")].copy()

# 매핑 딕셔너리 생성
gu_code_dict = dict(zip(seoul_gu["SIGUNGU_NM"], seoul_gu["SIGUNGU_CD"]))

# gu 정규화
def normalize_gu(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    # 혹시 "서울특별시 강남구" 형태라면 마지막 토큰만 사용
    if " " in x:
        x = x.split()[-1]
    return x

df["gu"] = df["gu"].apply(normalize_gu)

# 코드 매핑
df["COL_ADM_SE"] = df["gu"].map(gu_code_dict)

# 점검 출력
print("COL_ADM_SE 결측:", df["COL_ADM_SE"].isna().sum())

display(
    df[df["COL_ADM_SE"].isna()][
        ["name","address_std","gu"]
    ].head(20)
)

COL_ADM_SE 결측: 2


,name,address_std,gu
177,구로디지털1인샵[휴] ♡ 손 끝으로 전하는 감동 ■// 건식 //■// 림프순환테라...,서울시 동작구 신대방동 719 신대방1가길 38,NaN
218,"홈케어[1%명품] ❤️❤️24시❤️❤️ㅣ강남구ㅣ강남,논현,신사,선릉,삼성,역삼",None,NaN


In [35]:
df_false = df[df["is_valid_location"] == False].copy()

print("is_valid_location=False 개수:", len(df_false))
print("그 중 address_std 존재:", df_false["address_std"].notna().sum())
print("그 중 station_exit 존재:", df_false["station_exit"].notna().sum())
print("그 중 station 존재:", df_false["station"].notna().sum())
print("그 중 lat 결측:", df_false["lat"].isna().sum())

display(df_false[["name","address_raw","address_std","station_detail","station_exit","station","lat","lng","gu"]].head(30))

is_valid_location=False 개수: 136
그 중 address_std 존재: 135
그 중 station_exit 존재: 109
그 중 station 존재: 111
그 중 lat 결측: 25


,name,address_raw,address_std,station_detail,station_exit,station,lat,lng,gu
7,동대문.제기1인샵[오닉스] ⚜️ New 신규샵 ⋆ 한국 관리사 ⋆ 친절마인드 ⚜️,서울특별시 동대문구 제기동 / 제기동역 5번출구 도보5분,서울특별시 동대문구 제기동,제기동역 5번출구 도보5분,제기동역 5번출구,제기동역,37.578176,127.034591,동대문구
15,구로.신도림1인샵[채린]♥▣▣▣ 릴렉싱&스웨디시 ▣▣ 따뜻한 힐링공간▣▣ 친절한 마...,서울특별시 구로구 신도림동 / 신도림역 5번출구 도보5분,서울특별시 구로구 신도림동,신도림역 5번출구 도보5분,신도림역 5번출구,신도림역,37.509201,126.882575,구로구
16,마포.상암1인샵[빛나는],서울특별시 마포구 상암동(월드컵북로)/푸르메재단 넥슨어린이재활병원 도보5분,서울특별시 마포구 상암동 월드컵북로,NaN,NaN,NaN,NaN,NaN,마포구
19,마포.상암[레옹테라피]⭐친절한 한국인관리사⭐깨끗한시설⭐각방샤워시설⭐,서울특별시 마포구 상암동 1590(월드컵북로 434)/상암동MBC신사옥 도보10분/...,서울특별시 마포구 상암동 1590 월드컵북로 434,NaN,NaN,NaN,NaN,NaN,마포구
23,중랑.상봉1인샵[모모왁싱]❤️⭐️❤️✔️오후 5시전 주간 1만원 할인이벤트 !!☎️...,서울특별시 중랑구 상봉동(망우로)/망우역 1번출구 도보3분,서울특별시 중랑구 상봉동 망우로,망우역 1번출구 도보3분,NaN,NaN,NaN,NaN,중랑구
24,마포.공덕1인샵[수정] ❤️신규오픈 ❤️꧁수정1인샵꧂✨천상의테라피✨꧂✨천사의 친절마인드✨,서울특별시 마포구 신공덕동 172 (백범로 205) / 공덕역 6번출구 도보2분,서울특별시 마포구 신공덕동 172 백범로 205,공덕역 6번출구 도보2분,공덕역 6번출구,공덕역,37.550426,126.963167,마포구
26,서대문.충정로역1인샵[수아]☘️신규오픈☘️☀️세상에서 제일 친절한 마인드☀️,서울특별시 서대문구 충정로3가/서울특별시 서대문구 서소문로/충정로역 3번출구 도보5분,서울특별시 서대문구 충정로3가,충정로역 3번출구 도보5분,충정로역 3번출구,충정로역,37.562118,126.961955,서대문구
28,마포.공덕역1인샵[송민]❤️신규오픈 ❤️공덕역-송민1인샵✨천사의 친절마인드✨,서울특별시 마포구 도화동/서울특별시 마포구 마포대로/공덕역 1번출구 도보4분,서울특별시 마포구 마포대로,공덕역 1번출구 도보4분,공덕역 1번출구,공덕역,37.544503,126.949255,마포구
31,종로.종각역[블라]테라피 ♥❤♥ 힐링의 시간 ♥❤♥ 첫타임/첫방문 이벤트중^^,서울 종로구 종로1가 / 종각역 1번출구 도보5분 / 광화문역 4번출구 도보5분,서울특별시 종로구 종로1가,종각역 1번출구 도보5분,종각역 1번출구,종각역,37.570176,126.983197,종로구
32,용산역1인샵 오로라,서울특별시 용산구 한강로2가 420 (한강대로 69) / 용산역 1번출구 도보3분 ...,서울특별시 용산구 한강로2가 420 한강대로 69,용산역 1번출구 도보3분,NaN,NaN,NaN,NaN,용산구


In [36]:
# sigu 생성

def build_sigu(gu):
    if pd.isna(gu):
        return np.nan
    gu = str(gu).strip()
    if not gu:
        return np.nan
    return f"서울특별시 {gu}"

df["sigu"] = df["gu"].apply(build_sigu)

# 점검
print("sigu 결측:", df["sigu"].isna().sum())
display(df[["gu","sigu"]].drop_duplicates().sort_values("gu").head(20))

sigu 결측: 2


,gu,sigu
197,강남구,서울특별시 강남구
61,강동구,서울특별시 강동구
84,강북구,서울특별시 강북구
4,강서구,서울특별시 강서구
1,관악구,서울특별시 관악구
8,광진구,서울특별시 광진구
9,구로구,서울특별시 구로구
6,금천구,서울특별시 금천구
69,노원구,서울특별시 노원구
116,도봉구,서울특별시 도봉구


In [42]:
# ============ 업소명 및 카테고리 추가 정제 =============
CATEGORY_KEYWORDS = [
    "1인샵",
    "스웨디시",
    "아로마",
    "테라피",
    "로미로미",
    "마사지"
]

def clean_basic(text):
    if pd.isna(text):
        return None
    
    text = str(text)
    
    # 이모지 제거
    text = re.sub(r"[⭐✨❤️❣️⚡⚜️♥ღ✴️▣]+", " ", text)
    
    # 따옴표, 특수기호 일부 제거
    text = text.replace('"', " ")
    
    # 공백 정리
    text = re.sub(r"\s+", " ", text).strip()
    
    return text


def parse_name_category(text):
    if pd.isna(text):
        return None, None
    
    text = clean_basic(text)
    
    # 1️⃣ 대괄호 안 우선
    bracket = re.search(r"\[(.*?)\]", text)
    if bracket:
        name = bracket.group(1).strip()
        base_text = text.replace(bracket.group(0), "")
    else:
        tokens = text.split()
        name = tokens[-1]
        base_text = " ".join(tokens[:-1])
    
    # 2️⃣ 카테고리 키워드 추출
    categories = []
    for kw in CATEGORY_KEYWORDS:
        if kw in text:
            categories.append(kw)
    
    categories = list(set(categories))
    
    category = " | ".join(sorted(categories)) if categories else None
    
    return name, category


df[["name", "category"]] = (
    df["name_raw"]
    .apply(lambda x: pd.Series(parse_name_category(x)))
)

In [43]:
clean_cols = [
    "name",
    "category",
    "address_std",
    "gu",
    "sigu",
    "lat",
    "lng",
    "is_valid_location",
    "COL_ADM_SE",
    "station",
    "station_exit",
    "station_detail",
]

df_clean = df[clean_cols].copy()

OUT = "../data/clean_naviya.csv"
df_clean.to_csv(OUT, index=False, encoding="utf-8-sig")

print("Saved:", OUT)
display(df_clean.head(20))

Saved: ../data/clean_naviya.csv


,name,category,address_std,gu,sigu,lat,lng,is_valid_location,COL_ADM_SE,station,station_exit,station_detail
0,노블테라피,테라피,서울특별시 양천구 신정동 1031-9 중앙로 262,양천구,서울특별시 양천구,37.519526,126.852866,True,11150,신정네거리역,신정네거리역 4번출구,신정네거리역 4번출구 도보1분
1,청담스웨디시,스웨디시,서울특별시 관악구 과천대로 947-2,관악구,서울특별시 관악구,37.475195,126.981348,True,11210,사당역,사당역 4번출구,사당역 4번출구 바로앞 도보1분
2,NOW,None,서울특별시 마포구 도화동 1-12 새창로 40,마포구,서울특별시 마포구,37.541241,126.954050,True,11140,공덕역,공덕역 10번출구,공덕역 10번출구 도보2분
3,온결,1인샵,서울특별시 성동구 도선동 58-1 왕십리로 326,성동구,서울특별시 성동구,37.566611,127.029906,True,11040,왕십리역,왕십리역 1번출구,왕십리역 1번출구 도보5분 (홈샵: 서울)
4,정스웨디시,스웨디시,서울특별시 강서구 마곡동 797-10 마곡동로4길 15,강서구,서울특별시 강서구,37.559832,126.835251,True,11160,발산역,발산역 9번출구,발산역 9번출구 도보4분
5,에르메스,1인샵,서울특별시 중랑구 망우로 346,중랑구,서울특별시 중랑구,37.597103,127.091182,True,11070,상봉역,상봉역 2번출구,상봉역 2번출구 도보4분
6,린 스웨디시,스웨디시,서울특별시 금천구 가산동 41-11 벚꽃로 312,금천구,서울특별시 금천구,37.482245,126.883013,True,11180,가산디지털단지역,가산디지털단지역 2번출구,가산디지털단지역 2번출구 도보2분
7,오닉스,1인샵,서울특별시 동대문구 제기동,동대문구,서울특별시 동대문구,37.578176,127.034591,False,11060,제기동역,제기동역 5번출구,제기동역 5번출구 도보5분
8,누구나아로마,아로마,서울특별시 광진구 뚝섬로 476,광진구,서울특별시 광진구,37.536454,127.063136,True,11050,NaN,NaN,NaN
9,레몬스웨디시,스웨디시 | 테라피,서울특별시 구로구 구로동 187-3,구로구,서울특별시 구로구,37.485683,126.899054,True,11170,구로디지털단지역,구로디지털단지역 3번출구,구로디지털단지역 3번출구 도보2분


In [44]:
df["name"].head(20)

0       노블테라피
1      청담스웨디시
2         NOW
3          온결
4       정스웨디시
5        에르메스
6      린 스웨디시
7         오닉스
8      누구나아로마
9      레몬스웨디시
10      샤인테라피
11         달빛
12    클로버스웨디시
13        베스파
14      리치테라피
15         채린
16        빛나는
17      퍼플테라피
18        물방울
19      레옹테라피
Name: name, dtype: object